# Statistical Methods in AI – Monsoon 2026

## Assignment-1

### Introduction

#### This Question is of using KNN to predict the score of a given essay text. Here we shall be creating KNN Regression model using various distances such as • Euclidean distance • Manhattan distance • Cosine distance for various K values. And further it will be evaluated using MAE,RMSE,R2,Pearson correlation coefficient.

### Reproducibility Setup

In [1]:
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
username = "vamsi.vutla"

seed = int(
    hashlib.sha256(username.encode()).hexdigest(),
    16
) % (2**32)

print("Seed:", seed)

Seed: 203204534


# Automated Essay Scoring using K-Nearest Neighbors

## Objective

The objective is to predict human-assigned essay scores using
handcrafted statistical features extracted directly from essay text
and a K-Nearest Neighbors regression model.

The pipeline consists of:

1. Dataset exploration
2. Handcrafted feature engineering
3. Train/validation/test splitting
4. Z-score normalization
5. KNN regression from scratch
6. Hyperparameter tuning
7. Evaluation using MAE, RMSE, R², and Pearson correlation

In [4]:
DATA_PATH = "../data/essays(in).csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print(df.head())

Dataset shape: (17307, 3)

Columns:
['essay_id', 'full_text', 'score']
  essay_id                                          full_text  score
0  000d118  Many people have car where they live. The thin...      3
1  000fe60  I am a scientist at NASA that is discussing th...      3
2  001ab80  People always wish they had the same technolog...      4
3  001bdc0  We all heard about Venus, the planet without a...      4
4  002ba53  Dear, State SenatorThis is a letter to argue i...      3


In [7]:
## now checking missing values 
df.info()
df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 17307 entries, 0 to 17306
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   essay_id   17307 non-null  str  
 1   full_text  17307 non-null  str  
 2   score      17307 non-null  int64
dtypes: int64(1), str(2)
memory usage: 405.8 KB


essay_id     0
full_text    0
score        0
dtype: int64

In [9]:
print("No of rows: ", len(df))
print("\nScore Distrubution: ")
print(df['score'].value_counts().sort_index())

No of rows:  17307

Score Distrubution: 
score
1    1252
2    4723
3    6280
4    3926
5     970
6     156
Name: count, dtype: int64


In [10]:
## loading one sample
print("Essay: ", df['full_text'][0])
print("\nScore: ",df['score'][0])

Essay:  Many people have car where they live. The thing they don't know is that when you use a car alot of thing can happen like you can get in accidet or the smoke that the car has is bad to breath on if someone is walk but in VAUBAN,Germany they dont have that proble because 70 percent of vauban's families do not own cars,and 57 percent sold a car to move there. Street parkig ,driveways and home garages are forbidden on the outskirts of freiburd that near the French and Swiss borders. You probaly won't see a car in Vauban's streets because they are completely "car free" but If some that lives in VAUBAN that owns a car ownership is allowed,but there are only two places that you can park a large garages at the edge of the development,where a car owner buys a space but it not cheap to buy one they sell the space for you car for $40,000 along with a home. The vauban people completed this in 2006 ,they said that this an example of a growing trend in Europe,The untile states and some where

#### Exploring the sample Essay

In [11]:
essay = df.loc[0, "full_text"]

print("Essay ID:", df.loc[0, "essay_id"])
print("Score:", df.loc[0, "score"])
print("Characters:", len(essay))

words = essay.split()

print("Whitespace-separated tokens:", len(words))
print("\nFirst 30 tokens:")
print(words[:30])

Essay ID: 000d118
Score: 3
Characters: 2677
Whitespace-separated tokens: 498

First 30 tokens:
['Many', 'people', 'have', 'car', 'where', 'they', 'live.', 'The', 'thing', 'they', "don't", 'know', 'is', 'that', 'when', 'you', 'use', 'a', 'car', 'alot', 'of', 'thing', 'can', 'happen', 'like', 'you', 'can', 'get', 'in', 'accidet']


In [13]:
import re

tokens = essay.split()

punctuation_tokens = [
    token for token in tokens
    if re.search(r"[^\w']", token)
]

print("Number of tokens containing punctuation:", len(punctuation_tokens))
print(punctuation_tokens)

Number of tokens containing punctuation: 33
['live.', 'VAUBAN,Germany', 'cars,and', 'there.', ',driveways', 'borders.', '"car', 'free"', 'allowed,but', 'development,where', '$40,000', 'home.', ',they', 'Europe,The', '"smart', 'planning".', 'States.', '5,500', 'mile.', '"All', 'cars,and', 'change"', 'car.', 'to.', ',the', '"car', 'reduced"communtunties,and', 'act,if', 'cautiously.', 'year.', 'bill,80', 'transports.', 'this.']


In [14]:
numeric_like = [
    token for token in tokens
    if any(char.isdigit() for char in token)
]

print("Tokens containing digits:", len(numeric_like))
print(numeric_like)

Tokens containing digits: 10
['70', '57', '$40,000', '2006', '12', '50', '5,500', '2', 'bill,80', '20']


In [15]:
apostrophe_tokens = [
    token for token in tokens
    if "'" in token
]

hyphen_tokens = [
    token for token in tokens
    if "-" in token
]

print("Apostrophe tokens:")
print(apostrophe_tokens[:50])

print("\nHyphen tokens:")
print(hyphen_tokens[:50])

Apostrophe tokens:
["don't", "vauban's", "won't", "Vauban's", "don't"]

Hyphen tokens:
[]


In [16]:
def tokenize_words(text):
    """
    Extract alphabetic words while preserving internal apostrophes.
    """

    words = re.findall(
        r"[A-Za-z]+(?:'[A-Za-z]+)?",
        text
    )

    return words

In [21]:
words = tokenize_words(essay)
words = [word.lower() for word in words]
print("Number of cleaned words: ",len(words))
print("Words: ",words)

Number of cleaned words:  498
Words:  ['many', 'people', 'have', 'car', 'where', 'they', 'live', 'the', 'thing', 'they', "don't", 'know', 'is', 'that', 'when', 'you', 'use', 'a', 'car', 'alot', 'of', 'thing', 'can', 'happen', 'like', 'you', 'can', 'get', 'in', 'accidet', 'or', 'the', 'smoke', 'that', 'the', 'car', 'has', 'is', 'bad', 'to', 'breath', 'on', 'if', 'someone', 'is', 'walk', 'but', 'in', 'vauban', 'germany', 'they', 'dont', 'have', 'that', 'proble', 'because', 'percent', 'of', "vauban's", 'families', 'do', 'not', 'own', 'cars', 'and', 'percent', 'sold', 'a', 'car', 'to', 'move', 'there', 'street', 'parkig', 'driveways', 'and', 'home', 'garages', 'are', 'forbidden', 'on', 'the', 'outskirts', 'of', 'freiburd', 'that', 'near', 'the', 'french', 'and', 'swiss', 'borders', 'you', 'probaly', "won't", 'see', 'a', 'car', 'in', "vauban's", 'streets', 'because', 'they', 'are', 'completely', 'car', 'free', 'but', 'if', 'some', 'that', 'lives', 'in', 'vauban', 'that', 'owns', 'a', 'car',

In [24]:
## calculating features for this sample
word_count = len(words) # feature 1
word_lengths = [
    sum(char.isalpha() for char in word)
    for word in words
] # feature 2
print("Word_lengths: ", word_lengths)   

mean_word_length = np.mean(word_lengths) #feature 3
word_length_std = np.std(word_lengths) #feature 4

print("Mean word lengths: ",mean_word_length)
print("Word length std: ",word_length_std)

Word_lengths:  [4, 6, 4, 3, 5, 4, 4, 3, 5, 4, 4, 4, 2, 4, 4, 3, 3, 1, 3, 4, 2, 5, 3, 6, 4, 3, 3, 3, 2, 7, 2, 3, 5, 4, 3, 3, 3, 2, 3, 2, 6, 2, 2, 7, 2, 4, 3, 2, 6, 7, 4, 4, 4, 4, 6, 7, 7, 2, 7, 8, 2, 3, 3, 4, 3, 7, 4, 1, 3, 2, 4, 5, 6, 6, 9, 3, 4, 7, 3, 9, 2, 3, 9, 2, 8, 4, 4, 3, 6, 3, 5, 7, 3, 7, 4, 3, 1, 3, 2, 7, 7, 7, 4, 3, 10, 3, 4, 3, 2, 4, 4, 5, 2, 6, 4, 4, 1, 3, 9, 2, 7, 3, 5, 3, 4, 3, 6, 4, 3, 3, 4, 1, 5, 7, 2, 3, 4, 2, 3, 11, 5, 1, 3, 5, 4, 1, 5, 3, 2, 3, 5, 2, 3, 3, 4, 4, 3, 5, 3, 3, 3, 3, 5, 4, 1, 4, 3, 6, 6, 9, 4, 2, 4, 4, 4, 4, 2, 7, 2, 1, 7, 5, 2, 6, 3, 6, 6, 3, 4, 5, 4, 3, 8, 4, 4, 4, 3, 4, 2, 6, 5, 8, 3, 7, 7, 2, 11, 6, 10, 3, 9, 4, 6, 3, 9, 4, 3, 11, 3, 7, 2, 10, 3, 9, 2, 6, 3, 2, 2, 7, 2, 4, 3, 9, 2, 3, 6, 6, 1, 8, 5, 4, 4, 4, 4, 4, 3, 4, 2, 6, 7, 4, 5, 6, 6, 3, 6, 3, 7, 3, 2, 6, 5, 3, 9, 6, 1, 11, 6, 4, 2, 3, 7, 5, 4, 4, 4, 4, 3, 2, 3, 11, 5, 5, 3, 3, 4, 8, 2, 3, 4, 3, 4, 4, 4, 2, 6, 3, 1, 5, 4, 3, 4, 4, 4, 5, 4, 4, 7, 4, 5, 2, 4, 4, 2, 2, 2, 3, 2, 6, 4, 4, 3, 4, 7, 4

In [25]:
short_words = sum(
    1 <= length <= 4
    for length in word_lengths
)

medium_words = sum(
    5 <= length <= 7
    for length in word_lengths
)

long_words = sum(
    length >= 8
    for length in word_lengths
)

print("Short_words: ",short_words)
print("Medium_words: ",medium_words)
print("Long_words: ",long_words)

Short_words:  340
Medium_words:  120
Long_words:  38


In [29]:
long_word_ratio = (
    long_words / word_count
    if word_count > 0
    else 0.0
)

short_word_ratio = (
    short_words / word_count
    if word_count > 0
    else 0.0
)

print("long_word_ratio: ", long_word_ratio)
print("short_word_ratio: ",short_word_ratio)

long_word_ratio:  0.07630522088353414
short_word_ratio:  0.6827309236947792


In [30]:
raw_sentences = re.split(r"[.!?]+", essay)

print("Number of raw sentence segments:", len(raw_sentences))

for i, sentence in enumerate(raw_sentences[:10]):
    print(f"\nSentence {i+1}:")
    print(sentence.strip())

Number of raw sentence segments: 14

Sentence 1:
Many people have car where they live

Sentence 2:
The thing they don't know is that when you use a car alot of thing can happen like you can get in accidet or the smoke that the car has is bad to breath on if someone is walk but in VAUBAN,Germany they dont have that proble because 70 percent of vauban's families do not own cars,and 57 percent sold a car to move there

Sentence 3:
Street parkig ,driveways and home garages are forbidden on the outskirts of freiburd that near the French and Swiss borders

Sentence 4:
You probaly won't see a car in Vauban's streets because they are completely "car free" but If some that lives in VAUBAN that owns a car ownership is allowed,but there are only two places that you can park a large garages at the edge of the development,where a car owner buys a space but it not cheap to buy one they sell the space for you car for $40,000 along with a home

Sentence 5:
The vauban people completed this in 2006 ,the

In [33]:
sentence_word_lengths = [
    len(tokenize_words(sentence))
    for sentence in sentences
]

print("Sentence word lengths: ", sentence_word_lengths)

Sentence word lengths:  [7, 65, 20, 74, 36, 36, 32, 127, 24, 25, 25, 18, 9]


In [34]:
sentence_count = len(sentences)

print("Sentence count: ", sentence_count)

mean_sentence_length = (
    np.mean(sentence_word_lengths)
    if sentence_word_lengths
    else 0
)

print("Sentence Mean length: ", mean_sentence_length)

sentence_length_std = (
    np.std(sentence_word_lengths)
    if sentence_word_lengths
    else 0
)

print("Sentence length deviation: ", sentence_length_std)

Sentence count:  13
Sentence Mean length:  38.30769230769231
Sentence length deviation:  31.682037170768247


In [35]:
short_sentences = sum(
    1 <= length <= 10
    for length in sentence_word_lengths
)

medium_sentences = sum(
    11 <= length <= 20
    for length in sentence_word_lengths
)

long_sentences = sum(
    length >= 21
    for length in sentence_word_lengths
)

long_sentence_ratio = (
    long_sentences / sentence_count
    if sentence_count > 0
    else 0.0
)

print(
    short_sentences
    + medium_sentences
    + long_sentences
)

print(sentence_count)

13
13


In [36]:
print("Short sentences:", short_sentences)
print("Medium sentences:", medium_sentences)
print("Long sentences:", long_sentences)

print("Long-sentence ratio:", long_sentence_ratio)

Short sentences: 2
Medium sentences: 2
Long sentences: 9
Long-sentence ratio: 0.6923076923076923


In [37]:
unique_words = len(set(words))

unique_word_ratio = (
    unique_words/word_count
    if word_count > 0
    else 0
)

print("Unique word ratio: ", unique_word_ratio)

Unique word ratio:  0.4457831325301205


In [38]:
numeric_tokens = [
    token for token in essay.split()
    if any(char.isdigit() for char in token)
]

numeric_token_count = len(numeric_tokens)

print("Numeric tokens:", numeric_tokens)
print("Numeric token count:", numeric_token_count)

Numeric tokens: ['70', '57', '$40,000', '2006', '12', '50', '5,500', '2', 'bill,80', '20']
Numeric token count: 10
